In [40]:
# --- Bibliotheques et reproductibilite ---
import matplotlib
matplotlib.use('Agg')  # backend non-interactif pour environnement headless
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

print("tensorflow :", tf.__version__, "| keras :", keras.__version__, "| numpy :", np.__version__)

tensorflow : 2.21.0 | keras : 3.15.1 | numpy : 2.5.3


In [41]:
# --- Constantes et dossier des donnees ---
from pathlib import Path

DATA = Path("../data_UC5") if Path.cwd().name == "notebooks" else Path("data_UC5")

RUL_MAX = 125        # la RUL est plafonnee a 125 cycles
SEUIL_ALERTE = 30    # une pompe est signalee si la RUL predite < 30
SEUIL_FA = 60        # fausse alerte si la RUL reelle > 60

CAPTEURS = [f"capteur_{i}" for i in range(8)]
FEATURES = CAPTEURS + ["charge"]   # 9 entrees = 8 capteurs + regime de charge

print("dossier data :", DATA.resolve())

dossier data : /home/moussl/PycharmProjects/ProjectFM/data_UC5


In [42]:
# --- Chargement des tableaux .npy et construction du tableau de travail ---
capteurs = np.load(DATA / "capteurs_train.npy")   # (N, 8)
charge = np.load(DATA / "charge_train.npy")        # (N,)
cycle = np.load(DATA / "cycle_train.npy")          # (N,)
pompe = np.load(DATA / "pompe_train.npy")          # (N,)
rul = np.load(DATA / "rul_train.npy")              # (N,)

df = pd.DataFrame({"pompe": pompe, "cycle": cycle, "charge": charge, "rul": rul})
df = pd.concat([df, pd.DataFrame(capteurs, columns=CAPTEURS)], axis=1)

print("df :", df.shape)

df : (2847, 12)


In [43]:
# --- Les 5 premieres lignes (un cycle par ligne) ---
df.head()

,pompe,cycle,charge,rul,capteur_0,capteur_1,capteur_2,capteur_3,capteur_4,capteur_5,capteur_6,capteur_7
0,501,1,0.8,287,64.121368,85.283852,5.703406,386.112061,2.799164,2.926858,60.545464,2940.054688
1,501,2,0.8,286,65.842484,88.605904,5.843378,382.737915,2.613886,3.232284,59.429459,2939.211670
2,501,3,0.6,285,61.027695,64.997971,4.149714,300.460449,2.635431,2.805421,59.821304,2951.151367
3,501,4,0.6,284,60.993416,65.993988,4.331102,295.942719,2.664095,2.942466,57.797825,2952.621094
4,501,5,0.6,283,61.448277,66.742538,3.861803,288.022522,2.255884,2.674360,59.544952,2945.199951


In [44]:
df.groupby("pompe")["cycle"].max()

pompe
501    288
502    318
503    337
504    155
505    386
506    288
507    369
508    180
509    151
510    375
Name: cycle, dtype: int16

In [45]:
df["charge"].value_counts()

charge
0.8    957
0.6    948
1.0    942
Name: count, dtype: int64

In [46]:
df[CAPTEURS].describe().T

,count,mean,std,min,25%,50%,75%,max
capteur_0,2847.0,64.259354,3.792441,55.135666,61.380371,64.066513,66.433197,80.946022
capteur_1,2847.0,87.004807,17.777519,55.372734,66.862804,87.048889,106.705402,120.426399
capteur_2,2847.0,5.720295,1.197269,2.913078,4.429073,5.762125,7.052977,7.832178
capteur_3,2847.0,381.375610,79.020126,210.143555,294.395859,383.246033,471.360504,515.012634
capteur_4,2847.0,2.972434,0.767904,2.066233,2.549812,2.729548,2.995620,6.782763
capteur_5,2847.0,3.224263,0.785955,1.964009,2.828746,3.061193,3.310765,7.960818
capteur_6,2847.0,62.292007,2.278130,56.621468,60.611128,62.327423,63.746716,71.312256
capteur_7,2847.0,2943.461914,4.821505,2926.181885,2940.097656,2943.570557,2946.876343,2957.728760


In [47]:
df[CAPTEURS].hist(figsize=(14, 7), bins=40)
plt.suptitle("Distribution des capteurs (train)")
plt.tight_layout()
plt.show()

/tmp/ipykernel_9518/67897110.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [48]:
# --- Correlation capteur / RUL ---
corr = df[CAPTEURS + ["rul"]].corr()["rul"].drop("rul").sort_values()

plt.figure(figsize=(8, 4))
sns.barplot(x=corr.values, y=corr.index, hue=corr.index, palette="vlag", legend=False)
for i, v in enumerate(corr.values):
    plt.text(v + (0.01 if v >= 0 else -0.01), i, f"{v:.2f}",
             va="center", ha="left" if v >= 0 else "right", fontsize=8)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Correlation capteur / RUL")
plt.xlabel("correlation de Pearson")
plt.tight_layout()
plt.show()

/tmp/ipykernel_9518/2959840883.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [49]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, c in zip(axes, CAPTEURS[:4]):
    sns.boxplot(data=df, x="charge", y=c, ax=ax)
plt.suptitle("Effet du regime de charge sur les 4 premiers capteurs")
plt.tight_layout()
plt.show()

/tmp/ipykernel_9518/3905384502.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [50]:
# --- RUL cap & cible (AVANT le split pour que train/val l'aient) ---
df["rul_cap"] = np.minimum(df["rul"], RUL_MAX)
df["cible"] = df["rul_cap"] / RUL_MAX

df[["pompe", "cycle", "rul", "rul_cap", "cible"]].head()

# --- Decoupage par pompe : 2 pompes en validation, 8 en entrainement ---
pompes_val = [502, 509]
train = df[~df["pompe"].isin(pompes_val)].copy()
val = df[df["pompe"].isin(pompes_val)].copy()

train = train.sort_values(["pompe", "cycle"]).reset_index(drop=True)
val = val.sort_values(["pompe", "cycle"]).reset_index(drop=True)

print("entrainement :", train.shape, "| validation :", val.shape)

entrainement : (2378, 14) | validation : (469, 14)


In [51]:
# --- Couches de normalisation Keras (apprises sur l'entrainement uniquement) ---
# Reference : une couche Normalization par regime, apprise sur les cycles sains du train.
norm_regime = {}
for c in [0.6, 0.8, 1.0]:
    sains = train[(train["charge"] == c) & (train["rul_cap"] == RUL_MAX)]
    norm_regime[c] = layers.Normalization(axis=-1)
    norm_regime[c].adapt(sains[CAPTEURS].values.astype("float32"))

norm_global = layers.Normalization(axis=-1)
norm_global.adapt(train[CAPTEURS].values.astype("float32"))

print("couches de normalisation pretes :", len(norm_regime), "par regime + 1 globale")

couches de normalisation pretes : 3 par regime + 1 globale


In [52]:
# --- Fonctions d'application de la normalisation ---
def normaliser(donnees):
    d = donnees.copy()
    for c, couche in norm_regime.items():
        lignes = d["charge"] == c
        d.loc[lignes, CAPTEURS] = couche(d.loc[lignes, CAPTEURS].values.astype("float32")).numpy()
    return d

def normaliser_global(donnees):
    d = donnees.copy()
    d[CAPTEURS] = norm_global(d[CAPTEURS].values.astype("float32")).numpy()
    return d

print("fonctions de normalisation definies")

fonctions de normalisation definies


In [53]:
# --- Entrees et cibles ---
train_n = normaliser(train)
val_n = normaliser(val)

X_train = train_n[FEATURES].values.astype("float32")
y_train = train_n["cible"].values.astype("float32")
X_val = val_n[FEATURES].values.astype("float32")
y_val = val_n["cible"].values.astype("float32")
y_val_cycles = val_n["rul_cap"].values.astype(float)

DECOUPES = [5, 15, 30, 45, 60, 90, 120]
masque_coupes = val_n["rul_cap"].isin(DECOUPES).values

print("X_train :", X_train.shape, "| X_val :", X_val.shape, "| points de coupe :", int(masque_coupes.sum()))

X_train : (2378, 9) | X_val : (469, 9) | points de coupe : 14


In [54]:
# --- Definition des metriques ---
def rmse(y_vrai, y_pred):
    return float(np.sqrt(np.mean((y_vrai - y_pred) ** 2)))

def score_nasa(y_vrai, y_pred):
    d = y_pred - y_vrai
    return float(np.sum(np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)))

def taux_alerte(y_vrai, y_pred):
    alerte = y_pred < SEUIL_ALERTE
    proches = y_vrai <= SEUIL_ALERTE
    lointaines = y_vrai > SEUIL_FA
    return float(alerte[proches].mean()), float(alerte[lointaines].mean())

def evaluer(modele, X):
    # predictions ramenees en cycles, bornees a [0, 125]
    pred = np.clip(RUL_MAX * modele.predict(X, verbose=0).ravel(), 0, RUL_MAX)
    a, f = taux_alerte(y_val_cycles[masque_coupes], pred[masque_coupes])
    return {"rmse": rmse(y_val_cycles, pred),
            "score": score_nasa(y_val_cycles, pred),
            "anticipation": a,
            "fausses_alertes": f}, pred

print("metriques definies : rmse, score_nasa, taux_alerte, evaluer")

metriques definies : rmse, score_nasa, taux_alerte, evaluer


In [55]:
# --- Architecture de reference (copier ce bloc dans chaque experience) ---
BaseModel = keras.Sequential([
    keras.Input(shape=(9,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
print("nombre de parametres :", BaseModel.count_params())

nombre de parametres : 2753


In [56]:
# --- Journal + suivi MLflow ---
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow_jalon1.db")
mlflow.set_experiment("uc5_jalon1")
journal = []

def journaliser(nom, parametres, resultats):
    journal.append({"jalon": 1, "nom": nom, **parametres, **resultats})
    pd.DataFrame(journal).to_csv("journal_jalon1.csv", index=False)
    with mlflow.start_run(run_name=nom):
        mlflow.log_params({k: str(v) for k, v in parametres.items()})
        mlflow.log_metrics({k: float(v) for k, v in resultats.items() if np.isfinite(v)})
    print(nom, "->", {k: round(v, 2) for k, v in resultats.items()})
    return journal[-1]

print("journal pret")

journal pret


In [57]:
# --- Q1 : entrainement du modele de reference (test rapide 2 epochs) ---
reference = keras.Sequential([
    keras.Input(shape=(X_train.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
reference.compile(optimizer="adam", loss="mse")
reference.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=2, batch_size=256, verbose=1)

res_ref, pred_ref = evaluer(reference, X_val)
journaliser("reference (MSE, regime, k=1)",
            {"normalisation": "regime", "k": 1, "perte": "mse"}, res_ref)

Epoch 1/2
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.1813 - val_loss: 0.1105
Epoch 2/2
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.1070 - val_loss: 0.0615
reference (MSE, regime, k=1) -> {'rmse': 31.0, 'score': 7272.65, 'anticipation': 1.0, 'fausses_alertes': 0.0}


{'jalon': 1,
 'nom': 'reference (MSE, regime, k=1)',
 'normalisation': 'regime',
 'k': 1,
 'perte': 'mse',
 'rmse': 30.998996102052413,
 'score': 7272.653309614647,
 'anticipation': 1.0,
 'fausses_alertes': 0.0}

In [58]:
# --- Q1 : erreur en fonction de la RUL reelle ---
ecart = pred_ref - y_val_cycles
plt.figure(figsize=(8, 4))
plt.scatter(y_val_cycles, ecart, s=4, alpha=0.4)
plt.axhline(0, color="red", linestyle=":")
plt.xlabel("RUL reelle (cycles)")
plt.ylabel("RUL predite - RUL reelle")
plt.title(" erreur du modele de reference")
plt.show()

/tmp/ipykernel_9518/4058513195.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [59]:
# --- Q1 : biais moyen (predite - reelle) ---
print("biais moyen :", round(float(ecart.mean()), 2), "cycles")

biais moyen : -21.52 cycles


In [60]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# --- Q2 : modele avec normalisation globale (Optimise pour petit dataset) ---
train_g = normaliser_global(train)
val_g = normaliser_global(val)

Xg_train = train_g[FEATURES].values.astype("float32")
yg_train = train_g["cible"].values.astype("float32")
Xg_val = val_g[FEATURES].values.astype("float32")
yg_val = val_g["cible"].values.astype("float32")

ModelGlobalNorm = keras.Sequential([
    keras.Input(shape=(Xg_train.shape[1],)),

    layers.Dense(32, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.Dropout(0.2),

    layers.Dense(16, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.Dropout(0.2),

    layers.Dense(1, activation="sigmoid")
])

ModelGlobalNorm.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mse"
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
)

ModelGlobalNorm.fit(
    Xg_train, yg_train,
    validation_data=(Xg_val, yg_val),
    epochs=150,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=0
)

res_global, _ = evaluer(ModelGlobalNorm, Xg_val)
journaliser("normalisation globale",
            {"normalisation": "globale", "k": 1, "perte": "mse", "regularisation": "L2+Dropout"},
            res_global)

Epoch 47: early stopping
Restoring model weights from the end of the best epoch: 32.
normalisation globale -> {'rmse': 14.22, 'score': 2682.08, 'anticipation': 1.0, 'fausses_alertes': 0.0}


{'jalon': 1,
 'nom': 'normalisation globale',
 'normalisation': 'globale',
 'k': 1,
 'perte': 'mse',
 'regularisation': 'L2+Dropout',
 'rmse': 14.223201909754689,
 'score': 2682.075098039683,
 'anticipation': 1.0,
 'fausses_alertes': 0.0}

In [61]:
# --- Q2 : meme modele optimise, mais avec normalisation par regime ---
ModelRegimeNorm = keras.Sequential([
    keras.Input(shape=(X_train.shape[1],)),

    layers.Dense(32, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.Dropout(0.2),

    layers.Dense(16, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.Dropout(0.2),

    layers.Dense(1, activation="sigmoid")
])

ModelRegimeNorm.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mse"
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
)

ModelRegimeNorm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=150,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=0
)

res_regime, _ = evaluer(ModelRegimeNorm, X_val)
journaliser("regime (optimise)",
            {"normalisation": "regime", "k": 1, "perte": "mse", "regularisation": "L2+Dropout"},
            res_regime)

Epoch 40: early stopping
Restoring model weights from the end of the best epoch: 25.
regime (optimise) -> {'rmse': 15.48, 'score': 3815.56, 'anticipation': 1.0, 'fausses_alertes': 0.0}


{'jalon': 1,
 'nom': 'regime (optimise)',
 'normalisation': 'regime',
 'k': 1,
 'perte': 'mse',
 'regularisation': 'L2+Dropout',
 'rmse': 15.47546172891438,
 'score': 3815.555801455975,
 'anticipation': 1.0,
 'fausses_alertes': 0.0}

In [62]:
# --- Q2 : comparatif normalisation globale vs regime (meme modele optimise) ---
pd.DataFrame({
    "normalisation": ["globale", "regime"],
    "RMSE": [res_global["rmse"], res_regime["rmse"]],
    "score": [res_global["score"], res_regime["score"]],
    "anticipation": [res_global["anticipation"], res_regime["anticipation"]],
    "fausses alertes": [res_global["fausses_alertes"], res_regime["fausses_alertes"]],
})

,normalisation,RMSE,score,anticipation,fausses alertes
0,globale,14.223202,2682.075098,1.0,0.0
1,regime,15.475462,3815.555801,1.0,0.0


In [63]:
# --- Q3 : perte asymetrique et entrainement ---
def perte_asymetrique(y_vrai, y_pred):
    d = (y_pred - y_vrai) * RUL_MAX
    return tf.reduce_mean(tf.where(d < 0, tf.exp(-d / 13) - 1, tf.exp(d / 10) - 1))

modele_asym = keras.Sequential([
    keras.Input(shape=(X_train.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
modele_asym.compile(optimizer="adam", loss=perte_asymetrique)
modele_asym.fit(X_train, y_train, validation_data=(X_val, y_val),
                epochs=150, batch_size=256, verbose=0)

res_asym, _ = evaluer(modele_asym, X_val)
journaliser("perte asymetrique",
            {"normalisation": "regime", "k": 1, "perte": "asymetrique"}, res_asym)

perte asymetrique -> {'rmse': 19.03, 'score': 2437.73, 'anticipation': 1.0, 'fausses_alertes': 0.0}


{'jalon': 1,
 'nom': 'perte asymetrique',
 'normalisation': 'regime',
 'k': 1,
 'perte': 'asymetrique',
 'rmse': 19.033190186647992,
 'score': 2437.729800427128,
 'anticipation': 1.0,
 'fausses_alertes': 0.0}

In [64]:
# --- Q3 : tableau comparatif MSE vs perte asymetrique ---
pd.DataFrame({
    "perte": ["MSE", "asymetrique"],
    "RMSE": [res_ref["rmse"], res_asym["rmse"]],
    "score": [res_ref["score"], res_asym["score"]],
    "anticipation": [res_ref["anticipation"], res_asym["anticipation"]],
    "fausses alertes": [res_ref["fausses_alertes"], res_asym["fausses_alertes"]],
})

,perte,RMSE,score,anticipation,fausses alertes
0,MSE,30.998996,7272.65331,1.0,0.0
1,asymetrique,19.033190,2437.72980,1.0,0.0


In [65]:
def fenetres(donnees, k):
    d = donnees.sort_values(["pompe", "cycle"]).reset_index(drop=True)
    X = []
    for _, g in d.groupby("pompe"):
        v = g[FEATURES].values.astype("float32")
        v = np.vstack([np.tile(v[0], (k - 1, 1)), v])
        for i in range(len(g)):
            X.append(v[i:i + k].ravel())
    return (np.array(X, dtype="float32"),
            d["rul_cap"].values.astype(float),
            d["cible"].values.astype("float32"))

k = 1
Xk_train, yk_cycles, yk_train = fenetres(train_n, k)
Xk_val, yk_val_cycles, yk_val = fenetres(val_n, k)

modele_k = keras.Sequential([
    keras.Input(shape=(Xk_train.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
modele_k.compile(optimizer="adam", loss="mse")
modele_k.fit(Xk_train, yk_train, validation_data=(Xk_val, yk_val),
             epochs=150, batch_size=256, verbose=0)

pred_k = np.clip(RUL_MAX * modele_k.predict(Xk_val, verbose=0).ravel(), 0, RUL_MAX)
a, f = taux_alerte(yk_val_cycles[masque_coupes], pred_k[masque_coupes])
bruit = float(np.mean([pred_k[val_n["pompe"].values == p].std() for p in pompes_val]))
journaliser(f"fenetre k={k}", {"normalisation": "regime", "k": k, "perte": "mse"},
            {"rmse": rmse(yk_val_cycles, pred_k), "score": score_nasa(yk_val_cycles, pred_k),
             "anticipation": a, "fausses_alertes": f, "bruit": bruit})

fenetre k=1 -> {'rmse': 15.93, 'score': 3283.51, 'anticipation': 0.83, 'fausses_alertes': 0.0, 'bruit': 35.5}


{'jalon': 1,
 'nom': 'fenetre k=1',
 'normalisation': 'regime',
 'k': 1,
 'perte': 'mse',
 'rmse': 15.933540893922174,
 'score': 3283.506003172013,
 'anticipation': 0.8333333333333334,
 'fausses_alertes': 0.0,
 'bruit': 35.495513916015625}

In [66]:
# --- Q4 : k = 5 ---
k = 5
Xk_train, yk_cycles, yk_train = fenetres(train_n, k)
Xk_val, yk_val_cycles, yk_val = fenetres(val_n, k)

modele_k = keras.Sequential([
    keras.Input(shape=(Xk_train.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
modele_k.compile(optimizer="adam", loss="mse")
modele_k.fit(Xk_train, yk_train, validation_data=(Xk_val, yk_val),
             epochs=150, batch_size=256, verbose=0)

pred_k = np.clip(RUL_MAX * modele_k.predict(Xk_val, verbose=0).ravel(), 0, RUL_MAX)
a, f = taux_alerte(yk_val_cycles[masque_coupes], pred_k[masque_coupes])
bruit = float(np.mean([pred_k[val_n["pompe"].values == p].std() for p in pompes_val]))
journaliser(f"fenetre k={k}", {"normalisation": "regime", "k": k, "perte": "mse"},
            {"rmse": rmse(yk_val_cycles, pred_k), "score": score_nasa(yk_val_cycles, pred_k),
             "anticipation": a, "fausses_alertes": f, "bruit": bruit})

fenetre k=5 -> {'rmse': 16.8, 'score': 4135.98, 'anticipation': 0.67, 'fausses_alertes': 0.0, 'bruit': 36.21}


{'jalon': 1,
 'nom': 'fenetre k=5',
 'normalisation': 'regime',
 'k': 5,
 'perte': 'mse',
 'rmse': 16.797420595003313,
 'score': 4135.983138655306,
 'anticipation': 0.6666666666666666,
 'fausses_alertes': 0.0,
 'bruit': 36.20609664916992}

In [67]:
# --- Q4 : k = 15 ---
k = 15
Xk_train, yk_cycles, yk_train = fenetres(train_n, k)
Xk_val, yk_val_cycles, yk_val = fenetres(val_n, k)

modele_k = keras.Sequential([
    keras.Input(shape=(Xk_train.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
modele_k.compile(optimizer="adam", loss="mse")
modele_k.fit(Xk_train, yk_train, validation_data=(Xk_val, yk_val),
             epochs=150, batch_size=256, verbose=0)

pred_k = np.clip(RUL_MAX * modele_k.predict(Xk_val, verbose=0).ravel(), 0, RUL_MAX)
a, f = taux_alerte(yk_val_cycles[masque_coupes], pred_k[masque_coupes])
bruit = float(np.mean([pred_k[val_n["pompe"].values == p].std() for p in pompes_val]))
journaliser(f"fenetre k={k}", {"normalisation": "regime", "k": k, "perte": "mse"},
            {"rmse": rmse(yk_val_cycles, pred_k), "score": score_nasa(yk_val_cycles, pred_k),
             "anticipation": a, "fausses_alertes": f, "bruit": bruit})

fenetre k=15 -> {'rmse': 15.48, 'score': 4323.65, 'anticipation': 0.67, 'fausses_alertes': 0.0, 'bruit': 40.57}


{'jalon': 1,
 'nom': 'fenetre k=15',
 'normalisation': 'regime',
 'k': 15,
 'perte': 'mse',
 'rmse': 15.476565684227962,
 'score': 4323.64944908087,
 'anticipation': 0.6666666666666666,
 'fausses_alertes': 0.0,
 'bruit': 40.57288360595703}

In [68]:
# --- Q4 : k = 30 ---
k = 30
Xk_train, yk_cycles, yk_train = fenetres(train_n, k)
Xk_val, yk_val_cycles, yk_val = fenetres(val_n, k)

modele_k = keras.Sequential([
    keras.Input(shape=(Xk_train.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
modele_k.compile(optimizer="adam", loss="mse")
modele_k.fit(Xk_train, yk_train, validation_data=(Xk_val, yk_val),
             epochs=150, batch_size=256, verbose=0)

pred_k = np.clip(RUL_MAX * modele_k.predict(Xk_val, verbose=0).ravel(), 0, RUL_MAX)
a, f = taux_alerte(yk_val_cycles[masque_coupes], pred_k[masque_coupes])
bruit = float(np.mean([pred_k[val_n["pompe"].values == p].std() for p in pompes_val]))
journaliser(f"fenetre k={k}", {"normalisation": "regime", "k": k, "perte": "mse"},
            {"rmse": rmse(yk_val_cycles, pred_k), "score": score_nasa(yk_val_cycles, pred_k),
             "anticipation": a, "fausses_alertes": f, "bruit": bruit})

fenetre k=30 -> {'rmse': 15.88, 'score': 4502.64, 'anticipation': 0.83, 'fausses_alertes': 0.0, 'bruit': 41.31}


{'jalon': 1,
 'nom': 'fenetre k=30',
 'normalisation': 'regime',
 'k': 30,
 'perte': 'mse',
 'rmse': 15.88384440035991,
 'score': 4502.635121271785,
 'anticipation': 0.8333333333333334,
 'fausses_alertes': 0.0,
 'bruit': 41.308631896972656}

In [69]:
pd.DataFrame(journal)

,jalon,nom,normalisation,k,perte,rmse,score,anticipation,fausses_alertes,regularisation,bruit
0,1,"reference (MSE, regime, k=1)",regime,1,mse,30.998996,7272.653310,1.000000,0.0,NaN,NaN
1,1,normalisation globale,globale,1,mse,14.223202,2682.075098,1.000000,0.0,L2+Dropout,NaN
2,1,regime (optimise),regime,1,mse,15.475462,3815.555801,1.000000,0.0,L2+Dropout,NaN
3,1,perte asymetrique,regime,1,asymetrique,19.033190,2437.729800,1.000000,0.0,NaN,NaN
4,1,fenetre k=1,regime,1,mse,15.933541,3283.506003,0.833333,0.0,NaN,35.495514
5,1,fenetre k=5,regime,5,mse,16.797421,4135.983139,0.666667,0.0,NaN,36.206097
6,1,fenetre k=15,regime,15,mse,15.476566,4323.649449,0.666667,0.0,NaN,40.572884
7,1,fenetre k=30,regime,30,mse,15.883844,4502.635121,0.833333,0.0,NaN,41.308632


In [70]:
CAPTEURS_PEU_CORR = ["capteur_1", "capteur_2", "capteur_3", "capteur_7"]
FEATURES_5 = [f for f in FEATURES if f not in CAPTEURS_PEU_CORR]
FEATURES_3 = ["capteur_4", "capteur_5", "charge"]

GRAINES = [0, 1, 2]
ablation = []

def tester_entrees(nom, feats):
    rm, sc, an, fa = [], [], [], []
    for g in GRAINES:
        keras.utils.set_random_seed(g)
        Xt = train_n[feats].values.astype("float32")
        Xv = val_n[feats].values.astype("float32")
        modele = keras.Sequential([
            keras.Input(shape=(Xt.shape[1],)),
            layers.Dense(64, activation="relu"),
            layers.Dense(32, activation="relu"),
            layers.Dense(1, activation="sigmoid"),
        ])
        modele.compile(optimizer="adam", loss="mse")
        modele.fit(Xt, y_train, validation_data=(Xv, y_val),
                   epochs=150, batch_size=256, verbose=0)
        r, _ = evaluer(modele, Xv)
        rm.append(r["rmse"]); sc.append(r["score"])
        an.append(r["anticipation"]); fa.append(r["fausses_alertes"])
    res = {"rmse": float(np.mean(rm)), "score": float(np.mean(sc)),
           "anticipation": float(np.mean(an)), "fausses_alertes": float(np.mean(fa))}
    journaliser(f"capteurs : {nom}", {"normalisation": "regime", "k": 1, "perte": "mse",
                "nb_entrees": len(feats)}, res)
    ablation.append({"variante": nom, "entrees": len(feats), **res})

In [71]:
tester_entrees("9 entrees (reference)", FEATURES)

capteurs : 9 entrees (reference) -> {'rmse': 16.01, 'score': 3521.24, 'anticipation': 0.72, 'fausses_alertes': 0.0}


In [72]:
tester_entrees("5 entrees (sans capteurs |r| < 0.09)", FEATURES_5)

capteurs : 5 entrees (sans capteurs |r| < 0.09) -> {'rmse': 14.57, 'score': 2019.6, 'anticipation': 1.0, 'fausses_alertes': 0.0}


In [73]:
tester_entrees("3 entrees (capteur_4 + capteur_5 + charge)", FEATURES_3)

capteurs : 3 entrees (capteur_4 + capteur_5 + charge) -> {'rmse': 13.5, 'score': 1832.91, 'anticipation': 1.0, 'fausses_alertes': 0.0}


In [74]:
pd.DataFrame(ablation)

,variante,entrees,rmse,score,anticipation,fausses_alertes
0,9 entrees (reference),9,16.010966,3521.237733,0.722222,0.0
1,5 entrees (sans capteurs |r| < 0.09),5,14.570294,2019.600905,1.000000,0.0
2,3 entrees (capteur_4 + capteur_5 + charge),3,13.500167,1832.911536,1.000000,0.0
